In [ ]:
import os
from pathlib import Path

# Project folder
PROJECT_NAME = "nhs_waiting_list_forecasting"
BASE_DIR = Path.cwd() / PROJECT_NAME

# Subfolders
folders = [
    BASE_DIR / "data" / "raw",
    BASE_DIR / "data" / "processed",
    BASE_DIR / "outputs" / "figures",
    BASE_DIR / "outputs" / "tables",
    BASE_DIR / "notebooks"
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders created at:")
print(BASE_DIR)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose

print("Libraries loaded successfully.")

In [ ]:
rtt_xls_url = "https://www.england.nhs.uk/statistics/wp-content/uploads/sites/2/2026/03/RTT-Overview-Timeseries-Including-Estimates-for-Missing-Trusts-Jan26-XLS-116K-WL5BiP.xlsx"

rtt_book = pd.ExcelFile(rtt_xls_url)

print("Sheet names:")
print(rtt_book.sheet_names)

In [ ]:
rtt_df = pd.read_excel(rtt_xls_url, sheet_name="Full Time Series")

print(rtt_df.shape)
rtt_df.head(10)

In [ ]:
rtt_df.columns.tolist()

In [ ]:
rtt_df = pd.read_excel(rtt_xls_url, sheet_name="Full Time Series", header=9)

print(rtt_df.shape)
print(rtt_df.columns.tolist())
rtt_df.head()

In [ ]:
rtt_df = rtt_df.dropna(axis=1, how="all")

print(rtt_df.shape)
print(rtt_df.columns.tolist())
rtt_df.head()

In [ ]:
rtt_df = pd.read_excel(rtt_xls_url, sheet_name="Full Time Series", header=[9, 10])

rtt_df = rtt_df.dropna(axis=1, how="all")

rtt_df.columns = [
    " ".join(
        str(part).strip()
        for part in col
        if pd.notna(part) and "Unnamed" not in str(part)
    ).strip()
    for col in rtt_df.columns
]

rtt_df = rtt_df.loc[:, rtt_df.columns != ""]

print(rtt_df.shape)
print(rtt_df.columns.tolist())
rtt_df.head()

In [ ]:
[col for col in rtt_df.columns if "pathways" in col.lower() or "wait" in col.lower() or "month" in col.lower() or "year" in col.lower()]

In [ ]:
top_cols = rtt_df.columns.tolist()
sub_cols = rtt_df.iloc[0].tolist()

new_cols = []
seen = {}

for top, sub in zip(top_cols, sub_cols):
    top = "" if pd.isna(top) else str(top).strip()
    sub = "" if pd.isna(sub) else str(sub).strip()

    name_parts = [x for x in [top, sub] if x and x.lower() != "nan"]
    name = " ".join(name_parts).strip()

    if name == "":
        name = "blank_column"

    if name in seen:
        seen[name] += 1
        name = f"{name}_{seen[name]}"
    else:
        seen[name] = 0

    new_cols.append(name)

rtt_df.columns = new_cols
rtt_df = rtt_df.iloc[1:].reset_index(drop=True)

print(rtt_df.shape)
print(rtt_df.columns.tolist())
rtt_df.head()

In [ ]:
[c for c in rtt_df.columns if "incomplete rtt pathways" in c.lower() or c.lower() in ["year", "month"]]

In [ ]:
waiting_ts = rtt_df[[
    "Month",
    "Incomplete RTT pathways Total waiting (mil)"
]].copy()

waiting_ts = waiting_ts.rename(columns={
    "Month": "month",
    "Incomplete RTT pathways Total waiting (mil)": "waiting_mil"
})

waiting_ts["month"] = pd.to_datetime(waiting_ts["month"], errors="coerce")
waiting_ts["waiting_mil"] = pd.to_numeric(waiting_ts["waiting_mil"], errors="coerce")

waiting_ts = waiting_ts.dropna(subset=["month", "waiting_mil"]).sort_values("month").reset_index(drop=True)
waiting_ts["waiting_list"] = waiting_ts["waiting_mil"] * 1_000_000

print(waiting_ts.shape)
waiting_ts.head()

In [ ]:
waiting_ts.tail()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(waiting_ts["month"], waiting_ts["waiting_list"])
plt.title("England NHS RTT Waiting List")
plt.xlabel("Month")
plt.ylabel("Waiting List")
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(BASE_DIR / "outputs" / "figures" / "england_nhs_rtt_waiting_list.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
waiting_ts = rtt_df[[
    "Month",
    "Incomplete RTT pathways Total waiting (mil)"
]].copy()

waiting_ts = waiting_ts.rename(columns={
    "Month": "month",
    "Incomplete RTT pathways Total waiting (mil)": "waiting_list"
})

waiting_ts["month"] = pd.to_datetime(waiting_ts["month"], errors="coerce")
waiting_ts["waiting_list"] = pd.to_numeric(waiting_ts["waiting_list"], errors="coerce")

waiting_ts = (
    waiting_ts
    .dropna(subset=["month", "waiting_list"])
    .sort_values("month")
    .reset_index(drop=True)
)

print(waiting_ts.shape)
waiting_ts.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(waiting_ts["month"], waiting_ts["waiting_list"])
plt.title("England NHS RTT Waiting List")
plt.xlabel("Month")
plt.ylabel("Waiting List")
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(BASE_DIR / "outputs" / "figures" / "england_nhs_rtt_waiting_list_corrected.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
waiting_recent = waiting_ts[waiting_ts["month"] >= "2018-01-01"].copy().reset_index(drop=True)

print(waiting_recent.shape)
waiting_recent.head()

In [ ]:
waiting_recent.tail()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(waiting_recent["month"], waiting_recent["waiting_list"])
plt.title("England NHS RTT Waiting List (2018 onward)")
plt.xlabel("Month")
plt.ylabel("Waiting List")
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(BASE_DIR / "outputs" / "figures" / "england_nhs_rtt_waiting_list_2018_onward.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
decomp_input = waiting_recent.set_index("month")["waiting_list"].asfreq("MS")

print("Missing values:", decomp_input.isna().sum())
print(decomp_input[decomp_input.isna()])

In [ ]:
decomp_input = waiting_recent.set_index("month")["waiting_list"].asfreq("MS")
decomp_input = decomp_input.interpolate(method="linear")

print("Missing values after fix:", decomp_input.isna().sum())

In [ ]:
decomp = seasonal_decompose(decomp_input, model="additive", period=12)

fig = decomp.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.savefig(BASE_DIR / "outputs" / "figures" / "england_nhs_rtt_decomposition_2018_onward.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
forecast_input = waiting_recent.set_index("month")["waiting_list"].asfreq("MS")
forecast_input = forecast_input.interpolate(method="linear")

In [ ]:
hw_model = ExponentialSmoothing(
    forecast_input,
    trend="add",
    seasonal="add",
    seasonal_periods=12
).fit()

forecast_steps = 12
forecast_values = hw_model.forecast(forecast_steps)

forecast_df = pd.DataFrame({
    "month": pd.date_range(
        start=forecast_input.index[-1] + pd.offsets.MonthBegin(1),
        periods=forecast_steps,
        freq="MS"
    ),
    "forecast_waiting_list": forecast_values.values
})

forecast_df

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(waiting_recent["month"], waiting_recent["waiting_list"], label="Historical")
plt.plot(forecast_df["month"], forecast_df["forecast_waiting_list"], label="Forecast")
plt.title("England NHS RTT Waiting List Forecast (Next 12 Months)")
plt.xlabel("Month")
plt.ylabel("Waiting List")
plt.xticks(rotation=45)
plt.legend()

plt.tight_layout()
plt.savefig(BASE_DIR / "outputs" / "figures" / "england_nhs_rtt_forecast_12_months.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
model_series = waiting_recent.set_index("month")["waiting_list"].asfreq("MS")
model_series = model_series.interpolate(method="linear")

train = model_series.iloc[:-12]
test = model_series.iloc[-12:]

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train end:", train.index[-1])
print("Test start:", test.index[0])
print("Test end:", test.index[-1])

In [ ]:
hw_test_model = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=12
).fit()

test_forecast = hw_test_model.forecast(len(test))

test_results = pd.DataFrame({
    "month": test.index,
    "actual": test.values,
    "forecast": test_forecast.values
})

test_results.head()

In [ ]:
test_results["error"] = test_results["actual"] - test_results["forecast"]
test_results["abs_error"] = test_results["error"].abs()
test_results["sq_error"] = test_results["error"] ** 2
test_results["ape"] = (test_results["abs_error"] / test_results["actual"]) * 100

mae = test_results["abs_error"].mean()
rmse = np.sqrt(test_results["sq_error"].mean())
mape = test_results["ape"].mean()

print(f"MAE: {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"MAPE: {mape:.2f}%")

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train.index, train.values, label="Train")
plt.plot(test.index, test.values, label="Actual Test")
plt.plot(test_results["month"], test_results["forecast"], label="Forecast on Test")
plt.title("England NHS RTT Waiting List: Train/Test Forecast Evaluation")
plt.xlabel("Month")
plt.ylabel("Waiting List")
plt.xticks(rotation=45)
plt.legend()

plt.tight_layout()
plt.savefig(BASE_DIR / "outputs" / "figures" / "england_nhs_rtt_train_test_evaluation.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
test_results.to_csv(BASE_DIR / "outputs" / "tables" / "england_nhs_rtt_test_results.csv", index=False)
test_results

In [ ]:
worst_months = test_results.sort_values("abs_error", ascending=False).reset_index(drop=True)
worst_months.head(12)

In [ ]:
naive_forecast = pd.Series(train.iloc[-1], index=test.index)

naive_results = pd.DataFrame({
    "month": test.index,
    "actual": test.values,
    "forecast": naive_forecast.values
})

naive_results["error"] = naive_results["actual"] - naive_results["forecast"]
naive_results["abs_error"] = naive_results["error"].abs()
naive_results["sq_error"] = naive_results["error"] ** 2
naive_results["ape"] = (naive_results["abs_error"] / naive_results["actual"]) * 100

naive_mae = naive_results["abs_error"].mean()
naive_rmse = np.sqrt(naive_results["sq_error"].mean())
naive_mape = naive_results["ape"].mean()

comparison_df = pd.DataFrame({
    "model": ["Holt-Winters", "Naive"],
    "MAE": [mae, naive_mae],
    "RMSE": [rmse, naive_rmse],
    "MAPE": [mape, naive_mape]
})

comparison_df

In [ ]:
rtt_book.sheet_names

In [ ]:
[s for s in rtt_book.sheet_names if "special" in s.lower() or "treat" in s.lower() or "function" in s.lower() or "provider" in s.lower()]

In [ ]:
specialty_df = pd.DataFrame([
    ["General Surgery Service", 386068, 59.6, 25012, 50112, 90835],
    ["Urology Service", 389052, 62.8, 19618, 57706, 94892],
    ["Trauma and Orthopaedic Service", 836764, 57.9, 46693, 102356, 162835],
    ["Ear Nose and Throat Service", 587673, 53.4, 13018, 88225, 111875],
    ["Ophthalmology Service", 596635, 69.7, 53711, 107614, 176052],
    ["Oral Surgery Service", 317724, 52.4, 13545, 38966, 55878],
    ["Neurosurgical Service", 57289, 60.3, 2369, 7969, 11764],
    ["Plastic Surgery Service", 103068, 54.3, 12266, 10507, 22568],
    ["Cardiothoracic Surgery Service", 9041, 71.4, 1349, 1205, 2500],
    ["General Internal Medicine Service", 34818, 66.6, 880, 7355, 10685],
    ["Gastroenterology Service", 383007, 64.8, 17375, 54587, 99323],
    ["Cardiology Service", 383680, 63.9, 9715, 61724, 89900],
    ["Dermatology Service", 396171, 59.8, 11921, 97496, 113870],
    ["Respiratory Medicine Service", 184826, 69.4, 2227, 40695, 54050],
    ["Neurology Service", 216423, 57.0, 581, 32642, 41039],
    ["Rheumatology Service", 115918, 66.3, 1415, 26878, 30474],
    ["Elderly Medicine Service", 26332, 79.3, 500, 8812, 10539],
    ["Gynaecology Service", 565134, 56.9, 16011, 91215, 119573],
    ["Other - Medical Services", 616287, 66.1, 15747, 139407, 176624],
    ["Other - Mental Health Services", 2969, 77.0, 7, 941, 882],
    ["Other - Paediatric Services", 314575, 63.3, 10188, 61382, 78100],
    ["Other - Surgical Services", 481591, 62.2, 22960, 107685, 154704],
    ["Other - Other Services", 150913, 73.9, 4563, 51464, 58622],
], columns=[
    "treatment_function",
    "incomplete_pathways",
    "pct_within_18_weeks",
    "completed_admitted",
    "completed_non_admitted",
    "new_rtt_periods"
])

specialty_df

In [ ]:
top_backlog = specialty_df.sort_values("incomplete_pathways", ascending=False).reset_index(drop=True)
top_backlog.head(10)

In [ ]:
worst_18_week = specialty_df.sort_values("pct_within_18_weeks", ascending=True).reset_index(drop=True)
worst_18_week.head(10)

In [ ]:
top10_backlog = top_backlog.head(10).sort_values("incomplete_pathways", ascending=True)

plt.figure(figsize=(12, 6))
plt.barh(top10_backlog["treatment_function"], top10_backlog["incomplete_pathways"])
plt.title("Top 10 NHS RTT Specialty Backlogs (January 2026)")
plt.xlabel("Incomplete Pathways")
plt.ylabel("Treatment Function")

plt.tight_layout()
plt.savefig(BASE_DIR / "outputs" / "figures" / "specialty_top10_backlog_jan2026.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
worst10_18 = worst_18_week.head(10).sort_values("pct_within_18_weeks", ascending=True)

plt.figure(figsize=(12, 6))
plt.barh(worst10_18["treatment_function"], worst10_18["pct_within_18_weeks"])
plt.title("Weakest NHS RTT Specialties by % Within 18 Weeks (January 2026)")
plt.xlabel("% Within 18 Weeks")
plt.ylabel("Treatment Function")

plt.tight_layout()
plt.savefig(BASE_DIR / "outputs" / "figures" / "specialty_worst18weeks_jan2026.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(specialty_df["incomplete_pathways"], specialty_df["pct_within_18_weeks"])

for _, row in specialty_df.iterrows():
    if row["incomplete_pathways"] >= 500000 or row["pct_within_18_weeks"] <= 58:
        plt.annotate(
            row["treatment_function"],
            (row["incomplete_pathways"], row["pct_within_18_weeks"]),
            fontsize=8
        )

plt.title("NHS RTT Specialty Bottlenecks: Size vs 18-Week Performance")
plt.xlabel("Incomplete Pathways")
plt.ylabel("% Within 18 Weeks")

plt.tight_layout()
plt.savefig(BASE_DIR / "outputs" / "figures" / "specialty_bottleneck_scatter_jan2026.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
worst10_18 = worst_18_week.head(10).sort_values("pct_within_18_weeks", ascending=True)

plt.figure(figsize=(12, 6))
plt.barh(worst10_18["treatment_function"], worst10_18["pct_within_18_weeks"])
plt.title("Weakest NHS RTT Specialties by % Within 18 Weeks (January 2026)")
plt.xlabel("% Within 18 Weeks")
plt.ylabel("Treatment Function")
plt.gca().invert_yaxis()

plt.tight_layout()
plt.savefig(BASE_DIR / "outputs" / "figures" / "specialty_worst18weeks_jan2026_fixed.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
commissioner_url = "https://www.england.nhs.uk/statistics/wp-content/uploads/sites/2/2026/03/Incomplete-Commissioner-Jan26-XLSX-4M-WL5BiP.xlsx"
provider_url = "https://www.england.nhs.uk/statistics/wp-content/uploads/sites/2/2026/03/Incomplete-Provider-Jan26-XLSX-9M-WL5BiP.xlsx"

comm_book = pd.ExcelFile(commissioner_url)

print(comm_book.sheet_names)

In [ ]:
comm_region_raw = pd.read_excel(commissioner_url, sheet_name="Region", header=None)

print(comm_region_raw.shape)
comm_region_raw.head(12)

In [ ]:
comm_region_raw.iloc[:15, :12]

In [ ]:
comm_region = pd.read_excel(commissioner_url, sheet_name="Region", header=13)

comm_region = comm_region.dropna(axis=1, how="all")
comm_region = comm_region.loc[:, ~comm_region.columns.astype(str).str.contains("^Unnamed")]

print(comm_region.shape)
print(comm_region.columns.tolist())
comm_region.head()

In [ ]:
[c for c in comm_region.columns if "Region" in str(c) or "Treatment" in str(c) or "18" in str(c) or "52" in str(c) or "Total" in str(c) or "%" in str(c)]

In [ ]:
region_key = comm_region[[
    "Region Name",
    "Treatment Function Code",
    "Treatment Function",
    "Total number of incomplete pathways",
    "Total within 18 weeks",
    "% within 18 weeks",
    "Average (median) waiting time (in weeks)",
    "92nd percentile waiting time (in weeks)",
    "Total 52 plus weeks",
    "% 52 plus weeks"
]].copy()

numeric_cols = [
    "Total number of incomplete pathways",
    "Total within 18 weeks",
    "% within 18 weeks",
    "Average (median) waiting time (in weeks)",
    "92nd percentile waiting time (in weeks)",
    "Total 52 plus weeks",
    "% 52 plus weeks"
]

for col in numeric_cols:
    region_key[col] = pd.to_numeric(region_key[col], errors="coerce")

print(region_key.shape)
region_key.head()

In [ ]:
sorted(region_key["Region Name"].dropna().unique().tolist())

In [ ]:
sorted(region_key["Region Name"].dropna().unique().tolist())

In [ ]:
comm_national = pd.read_excel(commissioner_url, sheet_name="National", header=13)

comm_national = comm_national.dropna(axis=1, how="all")
comm_national = comm_national.loc[:, ~comm_national.columns.astype(str).str.contains("^Unnamed")]

print(comm_national.shape)
print(comm_national.columns.tolist())
comm_national.head()

In [ ]:
[c for c in comm_national.columns if "Treatment" in str(c) or "Total" in str(c) or "18" in str(c) or "52" in str(c) or "Average" in str(c) or "92nd" in str(c) or "%" in str(c)]

In [ ]:
national_key = comm_national[[
    "Treatment Function Code",
    "Treatment Function",
    "Total number of incomplete pathways",
    "Total within 18 weeks",
    "% within 18 weeks",
    "Average (median) waiting time (in weeks)",
    "92nd percentile waiting time (in weeks)",
    "Total 52 plus weeks",
    "% 52 plus weeks"
]].copy()

num_cols = [
    "Total number of incomplete pathways",
    "Total within 18 weeks",
    "% within 18 weeks",
    "Average (median) waiting time (in weeks)",
    "92nd percentile waiting time (in weeks)",
    "Total 52 plus weeks",
    "% 52 plus weeks"
]

for col in num_cols:
    national_key[col] = pd.to_numeric(national_key[col], errors="coerce")

print(national_key.shape)
national_key.head(10)

In [ ]:
national_compare = specialty_df.merge(
    national_key,
    left_on="treatment_function",
    right_on="Treatment Function",
    how="left"
)

national_compare["incomplete_ratio"] = (
    national_compare["Total number of incomplete pathways"] / national_compare["incomplete_pathways"]
)

national_compare[[
    "treatment_function",
    "incomplete_pathways",
    "Total number of incomplete pathways",
    "incomplete_ratio",
    "pct_within_18_weeks",
    "% within 18 weeks"
]].sort_values("incomplete_pathways", ascending=False)

In [ ]:
print(comm_icb.columns.tolist())

[c for c in comm_icb.columns if
 "region" in str(c).lower()
 or "icb" in str(c).lower()
 or "commissioner" in str(c).lower()
 or "treatment" in str(c).lower()
 or "total" in str(c).lower()
 or "18" in str(c).lower()
 or "52" in str(c).lower()
 or "%" in str(c)
]

In [ ]:
icb_key = comm_icb[[
    "ICB Code",
    "ICB Name",
    "Treatment Function Code",
    "Treatment Function",
    "Total number of incomplete pathways",
    "Total within 18 weeks",
    "% within 18 weeks",
    "Average (median) waiting time (in weeks)",
    "92nd percentile waiting time (in weeks)",
    "Total 52 plus weeks",
    "% 52 plus weeks"
]].copy()

icb_num_cols = [
    "Total number of incomplete pathways",
    "Total within 18 weeks",
    "% within 18 weeks",
    "Average (median) waiting time (in weeks)",
    "92nd percentile waiting time (in weeks)",
    "Total 52 plus weeks",
    "% 52 plus weeks"
]

for col in icb_num_cols:
    icb_key[col] = pd.to_numeric(icb_key[col], errors="coerce")

print(icb_key.shape)
icb_key.head()

In [ ]:
icb_to_national = (
    icb_key.groupby(["Treatment Function Code", "Treatment Function"], as_index=False)[
        ["Total number of incomplete pathways", "Total within 18 weeks", "Total 52 plus weeks"]
    ]
    .sum()
)

icb_validation = national_key.merge(
    icb_to_national,
    on=["Treatment Function Code", "Treatment Function"],
    suffixes=("_national", "_icb_sum"),
    how="left"
)

icb_validation["incomplete_ratio"] = (
    icb_validation["Total number of incomplete pathways_icb_sum"] /
    icb_validation["Total number of incomplete pathways_national"]
)

icb_validation["within18_ratio"] = (
    icb_validation["Total within 18 weeks_icb_sum"] /
    icb_validation["Total within 18 weeks_national"]
)

icb_validation["plus52_ratio"] = (
    icb_validation["Total 52 plus weeks_icb_sum"] /
    icb_validation["Total 52 plus weeks_national"]
)

icb_validation[[
    "Treatment Function",
    "Total number of incomplete pathways_national",
    "Total number of incomplete pathways_icb_sum",
    "incomplete_ratio",
    "within18_ratio",
    "plus52_ratio"
]].sort_values("incomplete_ratio", ascending=False)

In [ ]:
sorted(icb_key["ICB Name"].dropna().unique().tolist())

In [ ]:
icb_validation[[
    "Treatment Function",
    "Total number of incomplete pathways_national",
    "Total number of incomplete pathways_icb_sum",
    "incomplete_ratio",
    "within18_ratio",
    "plus52_ratio"
]].sort_values("incomplete_ratio", ascending=False)

In [ ]:
icb_working = icb_key[icb_key["ICB Name"] != "NHS ENGLAND"].copy()

print(icb_working.shape)
icb_working.head()

In [ ]:
icb_summary = (
    icb_working.groupby("ICB Name", as_index=False)
    .agg({
        "Total number of incomplete pathways": "sum",
        "Total within 18 weeks": "sum",
        "Total 52 plus weeks": "sum"
    })
)

icb_summary["pct_within_18_weeks"] = (
    icb_summary["Total within 18 weeks"] / icb_summary["Total number of incomplete pathways"]
)

icb_summary["pct_52_plus_weeks"] = (
    icb_summary["Total 52 plus weeks"] / icb_summary["Total number of incomplete pathways"]
)

icb_summary.head()

In [ ]:
top_icb_backlog = icb_summary.sort_values("Total number of incomplete pathways", ascending=False).reset_index(drop=True)
top_icb_backlog.head(10)

In [ ]:
worst_icb_18 = icb_summary.sort_values("pct_within_18_weeks", ascending=True).reset_index(drop=True)
worst_icb_18.head(10)

In [ ]:
plot_backlog = top_icb_backlog.head(10).sort_values("Total number of incomplete pathways", ascending=True)

plt.figure(figsize=(12, 6))
plt.barh(plot_backlog["ICB Name"], plot_backlog["Total number of incomplete pathways"])
plt.title("Top 10 ICBs by Total Incomplete Pathways")
plt.xlabel("Total Incomplete Pathways")
plt.ylabel("ICB Name")

plt.tight_layout()
plt.savefig(BASE_DIR / "outputs" / "figures" / "icb_top10_backlog.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plot_worst18 = worst_icb_18.head(10).sort_values("pct_within_18_weeks", ascending=True)

plt.figure(figsize=(12, 6))
plt.barh(plot_worst18["ICB Name"], plot_worst18["pct_within_18_weeks"])
plt.title("Worst 10 ICBs by % Within 18 Weeks")
plt.xlabel("% Within 18 Weeks")
plt.ylabel("ICB Name")
plt.gca().invert_yaxis()

plt.tight_layout()
plt.savefig(BASE_DIR / "outputs" / "figures" / "icb_worst18weeks.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
icb_summary.to_csv(BASE_DIR / "outputs" / "tables" / "icb_summary_jan2026.csv", index=False)
icb_summary.head()

In [ ]:
icb_metrics = icb_summary[[
    "ICB Name",
    "Total number of incomplete pathways",
    "Total within 18 weeks",
    "Total 52 plus weeks",
    "pct_within_18_weeks",
    "pct_52_plus_weeks"
]].copy()

icb_metrics = icb_metrics.rename(columns={
    "Total number of incomplete pathways": "incomplete_pathways",
    "Total within 18 weeks": "within_18_weeks",
    "Total 52 plus weeks": "plus_52_weeks"
})

icb_metrics.head()

In [ ]:
icb_metrics["backlog_rank"] = icb_metrics["incomplete_pathways"].rank(ascending=False, method="min")
icb_metrics["worst_18wk_rank"] = icb_metrics["pct_within_18_weeks"].rank(ascending=True, method="min")
icb_metrics["worst_52wk_rank"] = icb_metrics["pct_52_plus_weeks"].rank(ascending=False, method="min")

icb_metrics.sort_values("backlog_rank").head(10)

In [ ]:
icb_metrics.to_csv(BASE_DIR / "outputs" / "tables" / "icb_waiting_metrics_jan2026.csv", index=False)

In [ ]:
BASE_DIR / "data" / "raw" / "sapeicb20222024.xlsx"

In [ ]:
from urllib.request import Request, urlopen

dataset_page = "https://www.ons.gov.uk/peoplepopulationandcommunity/populationandmigration/populationestimates/datasets/clinicalcommissioninggroupmidyearpopulationestimates"
file_url = "https://www.ons.gov.uk/file?uri=%2Fpeoplepopulationandcommunity%2Fpopulationandmigration%2Fpopulationestimates%2Fdatasets%2Fclinicalcommissioninggroupmidyearpopulationestimates%2Fmid2022revisednov2025tomid2024integratedcareboards2024geography%2Fsapeicb20222024.xlsx"

ons_icb_pop_path = BASE_DIR / "data" / "raw" / "sapeicb20222024.xlsx"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": dataset_page,
    "Accept": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,application/octet-stream,*/*"
}

req = Request(file_url, headers=headers)

with urlopen(req, timeout=60) as response:
    data = response.read()

with open(ons_icb_pop_path, "wb") as f:
    f.write(data)

print("Saved to:", ons_icb_pop_path)
print("Exists:", ons_icb_pop_path.exists())
print("Size (bytes):", ons_icb_pop_path.stat().st_size)
print("First 2 bytes:", data[:2])

In [ ]:
ons_pop_book = pd.ExcelFile(ons_icb_pop_path)
print(ons_pop_book.sheet_names)

In [ ]:
ons_pop_raw = pd.read_excel(ons_icb_pop_path, sheet_name="Mid-2024 ICB 2024", header=None)

print(ons_pop_raw.shape)
ons_pop_raw.iloc[:15, :12]

In [ ]:
ons_pop_raw.iloc[:25, :8]

In [ ]:
ons_pop = pd.read_excel(ons_icb_pop_path, sheet_name="Mid-2024 ICB 2024", header=3)

ons_pop = ons_pop.dropna(axis=1, how="all")
ons_pop = ons_pop.loc[:, ~ons_pop.columns.astype(str).str.contains("^Unnamed")]

print(ons_pop.shape)
print(ons_pop.columns.tolist())
ons_pop.head()

In [ ]:
ons_icb_pop = ons_pop[[
    "ICB 2024 Code",
    "ICB 2024 Name",
    "Total"
]].copy()

ons_icb_pop["Total"] = pd.to_numeric(ons_icb_pop["Total"], errors="coerce")

print(ons_icb_pop.shape)
ons_icb_pop.head()

In [ ]:
ons_icb_pop_agg = (
    ons_icb_pop.groupby(["ICB 2024 Code", "ICB 2024 Name"], as_index=False)["Total"]
    .sum()
    .rename(columns={
        "ICB 2024 Code": "ICB Code",
        "ICB 2024 Name": "ICB Name",
        "Total": "population_2024"
    })
)

print(ons_icb_pop_agg.shape)
ons_icb_pop_agg.head()

In [ ]:
icb_summary_code = (
    icb_working.groupby(["ICB Code", "ICB Name"], as_index=False)
    .agg({
        "Total number of incomplete pathways": "sum",
        "Total within 18 weeks": "sum",
        "Total 52 plus weeks": "sum"
    })
)

icb_summary_code["pct_within_18_weeks"] = (
    icb_summary_code["Total within 18 weeks"] / icb_summary_code["Total number of incomplete pathways"]
)

icb_summary_code["pct_52_plus_weeks"] = (
    icb_summary_code["Total 52 plus weeks"] / icb_summary_code["Total number of incomplete pathways"]
)

print(icb_summary_code.shape)
icb_summary_code.head()

In [ ]:
icb_pop_merged = icb_summary_code.merge(
    ons_icb_pop_agg,
    on=["ICB Code", "ICB Name"],
    how="left"
)

print(icb_pop_merged.shape)
print("Missing population rows:", icb_pop_merged["population_2024"].isna().sum())
icb_pop_merged.head()

In [ ]:
icb_pop_merged["backlog_per_100k"] = (
    icb_pop_merged["Total number of incomplete pathways"] / icb_pop_merged["population_2024"]
) * 100000

icb_pop_merged["plus_52_per_100k"] = (
    icb_pop_merged["Total 52 plus weeks"] / icb_pop_merged["population_2024"]
) * 100000

icb_pop_merged[[
    "ICB Name",
    "population_2024",
    "Total number of incomplete pathways",
    "backlog_per_100k",
    "Total 52 plus weeks",
    "plus_52_per_100k",
    "pct_within_18_weeks"
]].sort_values("backlog_per_100k", ascending=False).head(10)

In [ ]:
icb_waiting_for_merge = icb_summary_code.copy()
ons_pop_for_merge = ons_icb_pop_agg.copy()

icb_waiting_for_merge["name_key"] = (
    icb_waiting_for_merge["ICB Name"]
    .astype(str)
    .str.lower()
    .str.replace("&", "and", regex=False)
    .str.replace("-", " ", regex=False)
    .str.replace(",", " ", regex=False)
    .str.replace(r"\bnhs\b", "", regex=True)
    .str.replace(r"\bintegrated care board\b", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

ons_pop_for_merge["name_key"] = (
    ons_pop_for_merge["ICB Name"]
    .astype(str)
    .str.lower()
    .str.replace("&", "and", regex=False)
    .str.replace("-", " ", regex=False)
    .str.replace(",", " ", regex=False)
    .str.replace(r"\bnhs\b", "", regex=True)
    .str.replace(r"\bintegrated care board\b", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

icb_pop_merged = icb_waiting_for_merge.merge(
    ons_pop_for_merge[["ICB Name", "population_2024", "name_key"]],
    on="name_key",
    how="left",
    suffixes=("_waiting", "_ons")
)

print(icb_pop_merged.shape)
print("Missing population rows:", icb_pop_merged["population_2024"].isna().sum())

icb_pop_merged[[
    "ICB Code",
    "ICB Name_waiting",
    "ICB Name_ons",
    "population_2024"
]].head()

In [ ]:
icb_pop_merged.loc[
    icb_pop_merged["population_2024"].isna(),
    ["ICB Code", "ICB Name_waiting", "name_key"]
].sort_values("ICB Name_waiting")

In [ ]:
icb_pop_merged["backlog_per_100k"] = (
    icb_pop_merged["Total number of incomplete pathways"] / icb_pop_merged["population_2024"]
) * 100000

icb_pop_merged["plus_52_per_100k"] = (
    icb_pop_merged["Total 52 plus weeks"] / icb_pop_merged["population_2024"]
) * 100000

icb_pop_merged[[
    "ICB Name_waiting",
    "population_2024",
    "Total number of incomplete pathways",
    "backlog_per_100k",
    "Total 52 plus weeks",
    "plus_52_per_100k",
    "pct_within_18_weeks"
]].sort_values("backlog_per_100k", ascending=False).head(10)

In [ ]:
icb_pop_merged["backlog_per_100k"] = (
    icb_pop_merged["Total number of incomplete pathways"] / icb_pop_merged["population_2024"]
) * 100000

icb_pop_merged["plus_52_per_100k"] = (
    icb_pop_merged["Total 52 plus weeks"] / icb_pop_merged["population_2024"]
) * 100000

icb_pop_merged["within_18_per_100k"] = (
    icb_pop_merged["Total within 18 weeks"] / icb_pop_merged["population_2024"]
) * 100000

icb_pop_merged[[
    "ICB Code",
    "ICB Name_waiting",
    "population_2024",
    "Total number of incomplete pathways",
    "backlog_per_100k",
    "Total 52 plus weeks",
    "plus_52_per_100k",
    "pct_within_18_weeks"
]].sort_values("backlog_per_100k", ascending=False).head(10)

In [ ]:
icb_final = icb_pop_merged[[
    "ICB Code",
    "ICB Name_waiting",
    "population_2024",
    "Total number of incomplete pathways",
    "Total within 18 weeks",
    "Total 52 plus weeks",
    "pct_within_18_weeks",
    "pct_52_plus_weeks",
    "backlog_per_100k",
    "plus_52_per_100k",
    "within_18_per_100k"
]].copy()

icb_final = icb_final.rename(columns={
    "ICB Name_waiting": "ICB Name",
    "Total number of incomplete pathways": "incomplete_pathways",
    "Total within 18 weeks": "within_18_weeks",
    "Total 52 plus weeks": "plus_52_plus_weeks"
})

icb_final.to_csv(BASE_DIR / "outputs" / "tables" / "icb_final_waiting_population_metrics.csv", index=False)

print(icb_final.shape)
icb_final.head()

In [ ]:
plot_backlog_per100k = icb_final.sort_values("backlog_per_100k", ascending=False).head(10)
plot_backlog_per100k = plot_backlog_per100k.sort_values("backlog_per_100k", ascending=True)

plt.figure(figsize=(12, 6))
plt.barh(plot_backlog_per100k["ICB Name"], plot_backlog_per100k["backlog_per_100k"])
plt.title("Top 10 ICBs by Backlog per 100,000 Population")
plt.xlabel("Backlog per 100,000")
plt.ylabel("ICB Name")

plt.tight_layout()
plt.savefig(BASE_DIR / "outputs" / "figures" / "icb_top10_backlog_per100k.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plot_52_per100k = icb_final.sort_values("plus_52_per_100k", ascending=False).head(10)
plot_52_per100k = plot_52_per100k.sort_values("plus_52_per_100k", ascending=True)

plt.figure(figsize=(12, 6))
plt.barh(plot_52_per100k["ICB Name"], plot_52_per100k["plus_52_per_100k"])
plt.title("Top 10 ICBs by 52+ Weeks per 100,000 Population")
plt.xlabel("52+ Weeks per 100,000")
plt.ylabel("ICB Name")

plt.tight_layout()
plt.savefig(BASE_DIR / "outputs" / "figures" / "icb_top10_52plus_per100k.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(icb_final["backlog_per_100k"], icb_final["pct_within_18_weeks"])

for _, row in icb_final.iterrows():
    if row["backlog_per_100k"] >= icb_final["backlog_per_100k"].quantile(0.85) or row["pct_within_18_weeks"] <= icb_final["pct_within_18_weeks"].quantile(0.15):
        plt.annotate(
            row["ICB Name"],
            (row["backlog_per_100k"], row["pct_within_18_weeks"]),
            fontsize=8
        )

plt.title("ICB Pressure: Backlog per 100k vs % Within 18 Weeks")
plt.xlabel("Backlog per 100,000")
plt.ylabel("% Within 18 Weeks")

plt.tight_layout()
plt.savefig(BASE_DIR / "outputs" / "figures" / "icb_backlog_per100k_vs_18weeks.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
icb_final.sort_values(["backlog_per_100k", "plus_52_per_100k"], ascending=[False, False]).head(15)

In [ ]:
project_summary = pd.DataFrame({
    "section": [
        "National forecast accuracy",
        "Top specialty backlog",
        "Weakest specialty by 18-week performance",
        "Top ICB by raw backlog",
        "Top ICB by backlog per 100k",
        "Top ICB by 52+ weeks per 100k"
    ],
    "result": [
        f"MAPE = {mape:.2f}%, MAE = {mae:,.0f}, RMSE = {rmse:,.0f}",
        "Trauma and Orthopaedic Service",
        "Oral Surgery Service",
        top_icb_backlog.iloc[0]['ICB Name'],
        icb_final.sort_values('backlog_per_100k', ascending=False).iloc[0]['ICB Name'],
        icb_final.sort_values('plus_52_per_100k', ascending=False).iloc[0]['ICB Name']
    ],
    "value": [
        "",
        int(top_backlog.iloc[0]["incomplete_pathways"]),
        float(worst_18_week.iloc[0]["pct_within_18_weeks"]),
        int(top_icb_backlog.iloc[0]["Total number of incomplete pathways"]),
        float(icb_final.sort_values("backlog_per_100k", ascending=False).iloc[0]["backlog_per_100k"]),
        float(icb_final.sort_values("plus_52_per_100k", ascending=False).iloc[0]["plus_52_per_100k"])
    ]
})

project_summary

In [ ]:
project_summary.to_csv(BASE_DIR / "outputs" / "tables" / "project_summary_table.csv", index=False)

In [ ]:
final_ranked_icbs = icb_final[[
    "ICB Code",
    "ICB Name",
    "population_2024",
    "incomplete_pathways",
    "backlog_per_100k",
    "plus_52_plus_weeks",
    "plus_52_per_100k",
    "pct_within_18_weeks",
    "pct_52_plus_weeks"
]].sort_values(
    ["backlog_per_100k", "plus_52_per_100k"],
    ascending=[False, False]
).reset_index(drop=True)

final_ranked_icbs.head(15)

In [ ]:
final_ranked_icbs.to_csv(BASE_DIR / "outputs" / "tables" / "final_ranked_icbs.csv", index=False)